# Doomscroll classifier: train + evaluate on Colab

Fine-tunes **Qwen2.5-1.5B-Instruct** with QLoRA, then compares it to a zero-shot baseline and a TF-IDF baseline.

**Before running:**
1. `Runtime > Change runtime type > T4 GPU`
2. Key icon (Secrets) in the left sidebar: add `HF_TOKEN` (a Hugging Face **write** token) and switch **Notebook access** on
3. Set `HF_REPO` in the next cell

Then `Runtime > Run all`. It takes about 15 to 25 minutes. The trained adapter is pushed to Hugging Face, so it survives the Colab session ending (v0 was lost because it only lived in `/content`).

In [ ]:
# >>> EDIT THIS <<<
HF_REPO = "your-username/doomscroll-qwen2.5-1.5b-lora"

REPO_URL = "https://github.com/SidharthaSagolsem/doomscroll_classifier.git"
assert not HF_REPO.startswith("your-username/"), "Set HF_REPO to <your HF username>/doomscroll-qwen2.5-1.5b-lora"

In [ ]:
# 1. Check the GPU
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then run again."
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Get the code (main branch)
%cd /content
!rm -rf doomscroll_classifier
!git clone --depth 1 -b main {REPO_URL}
%cd /content/doomscroll_classifier
!git log --oneline -1

In [ ]:
# 3. Install pinned training libraries (about 2 minutes)
!pip install -q -r requirements-train.txt

In [ ]:
# 4. Log in to Hugging Face with the HF_TOKEN secret (never printed)
import os
from google.colab import userdata
from huggingface_hub import login, whoami

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Logged in as:", whoami()["name"])

In [ ]:
# 5. Clean + split the data (dedup, audit, near-duplicate groups, stratified split)
!python prepare_data.py
import os
assert os.path.exists("data/processed/test.jsonl"), "prepare_data.py failed - see the error above"

In [ ]:
# 6. Train (QLoRA) and push the adapter to the Hub
!python train.py --output-dir outputs/lora --push-to-hub --hub-repo {HF_REPO}
import os
assert os.path.exists("outputs/lora/adapter_config.json"), "Training failed - see the error above. Fix it before running the next cells."

In [ ]:
# 7. Smoke test: the output should be one JSON object, not "Post Post Post..."
!python predict.py --model outputs/lora --text "POV: the rizzler hits the griddy in ohio at 3am fr fr skibidi" --text "Why does bread go stale? Starch molecules recrystallise and push water out, which is called retrogradation."

In [ ]:
# 8. Evaluate: fine-tuned vs zero-shot vs TF-IDF on the held-out test set
!python evaluate.py --model outputs/lora --systems finetuned,zero-shot,tfidf --out-dir results
import os
assert os.path.exists("results/report.md"), "evaluate.py failed - see the error above"

In [ ]:
# 9. Show the report
from IPython.display import Markdown, display
display(Markdown(open("results/report.md", encoding="utf-8").read()))

In [ ]:
# 10. Download results (metrics, predictions, training log; no model weights)
!cp outputs/lora/training_run.json results/
!cp data/processed/stats.json results/data_stats.json
!zip -q -r results.zip results
from google.colab import files
files.download("results.zip")
print(f"Model: https://huggingface.co/{HF_REPO}")